# Favorita Grocery Sales Forecasting

Forecasting daily store/product-family sales for the Kaggle [Store Sales -
Time Series Forecasting](https://www.kaggle.com/competitions/store-sales-time-series-forecasting)
(Favorita) competition, comparing a sliding-window LSTM against XGBoost,
and checking every claimed improvement with a walk-forward backtest before
believing it.

Full write-up and results table: see [`README.md`](./README.md). This
notebook is the reference run -- it calls into the `src/` package rather
than redefining logic inline, so the same functions can be reused,
re-tuned, or unit tested outside the notebook.

Data is loaded from a public Hugging Face mirror of the competition data
(no Kaggle credentials required) -- see `src/data.py`.


In [ ]:
import sys
sys.path.insert(0, ".")  # repo root, so `src` is importable when run from anywhere

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import data as data_mod
from src import features
from src import models
from src import tuning
from src import backtest
from src import pooling
from src import ensemble
from src.metrics import wape

pd.set_option("display.width", 120)


## 1. Data loading

Load train/stores/oil/holidays/transactions, and pick the single
highest-volume (store, family) series to develop the pipeline on before
generalizing to multiple series later.


In [ ]:
raw = data_mod.load_raw()
train, stores, oil, holidays, transactions = (
    raw["train"], raw["stores"], raw["oil"], raw["holidays"], raw["transactions"]
)
txn_available = transactions is not None

print(f"train: {train.shape[0]:,} rows, {train['date'].min().date()} to {train['date'].max().date()}")
print(f"{train['store_nbr'].nunique()} stores x {train['family'].nunique()} families")

vol, series_list_pooled = data_mod.top_volume_series(train, n=6)
top_store, top_family = series_list_pooled[0]
print(f"Top-volume series: store_nbr={top_store}, family='{top_family}'")
print(f"Pooling set (top 6 series): {series_list_pooled}")


## 2. A quick look at the data

Weekly seasonality, a promotion effect, and a holiday effect are the kind
of signal a forecasting model should be able to pick up -- worth checking
they're actually there before building anything.


In [ ]:
top_series = data_mod.build_series(train, oil, holidays, top_store, top_family)

s_eda = top_series.copy()
s_eda["day_of_week"] = s_eda["date"].dt.day_name()
dow_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

fig, axes = plt.subplots(2, 2, figsize=(13, 8))
fig.suptitle(f"Store {top_store}, family '{top_family}'", fontweight="bold")

axes[0, 0].plot(s_eda["date"], s_eda["sales"], linewidth=0.6, color="steelblue")
axes[0, 0].set_title("Daily sales, full history")

dow_means = s_eda.groupby("day_of_week")["sales"].mean().reindex(dow_order)
axes[0, 1].bar(range(7), dow_means.values, color="seagreen")
axes[0, 1].set_xticks(range(7))
axes[0, 1].set_xticklabels([d[:3] for d in dow_order])
axes[0, 1].set_title("Mean sales by day of week")

promo_group = s_eda.assign(has_promo=s_eda["onpromotion"] > 0).groupby("has_promo")["sales"].mean()
axes[1, 0].bar(["No promo", "On promo"], promo_group.values, color=["gray", "crimson"])
axes[1, 0].set_title("Promotion impact on mean sales")

hol_group = s_eda.groupby("is_holiday")["sales"].mean()
axes[1, 1].bar(["Non-holiday", "National holiday"], hol_group.values, color=["gray", "gold"])
axes[1, 1].set_title("Holiday impact on mean sales (naive national-only flag)")

plt.tight_layout()
plt.show()

print(f"On-promotion lift: {100*(promo_group[True]/promo_group[False]-1):.1f}%")
print(f"National-holiday lift: {100*(hol_group[1.0]/hol_group[0.0]-1):.1f}%")


## 3. Baseline: untuned sliding-window LSTM

A 13-feature baseline (sales lags, 7-day rolling mean/std, promotion,
national holiday flag, oil price, day-of-week and month Fourier terms),
fed through `TunableLSTM` with reasonable but untuned hyperparameters.
Benchmarked against two naive baselines: predict yesterday's sales, and
predict the same weekday last week.


In [ ]:
top_series_feat = features.engineer_features(top_series, features.BASE_FEATURE_COLS)

baseline_config = models.LSTMConfig(sequence_length=14, hidden_size=64, num_layers=2, dropout=0.2, lr=1e-3)
baseline_result = models.train_eval_lstm(top_series_feat, features.BASE_FEATURE_COLS, baseline_config)

actual = baseline_result["actual"]
naive_lag1 = top_series_feat["sales_lag1"].values[-len(actual):]
naive_lag7 = top_series_feat["sales_lag7"].values[-len(actual):]

print(f"Untuned LSTM WAPE:        {baseline_result['wape']:.2f}%")
print(f"Naive (predict lag1) WAPE: {wape(actual, naive_lag1):.2f}%")
print(f"Naive (predict lag7) WAPE: {wape(actual, naive_lag7):.2f}%")


## 4. Hyperparameter tuning (Optuna)

Search sequence length, hidden size, depth, dropout, and learning rate
with a seeded TPE sampler -- seeding the *sampler*, not just the model
weights, is what makes the search path (and therefore the "best"
hyperparameters it lands on) reproducible across runs.


In [ ]:
split_idx = int(len(top_series_feat) * 0.8)
from sklearn.preprocessing import StandardScaler

Xr = top_series_feat[features.BASE_FEATURE_COLS].values.astype("float32")
yr = top_series_feat["sales"].values.astype("float32").reshape(-1, 1)
x_scaler = StandardScaler().fit(Xr[:split_idx])
y_scaler = StandardScaler().fit(yr[:split_idx])
X_scaled, y_scaled = x_scaler.transform(Xr), y_scaler.transform(yr)

lstm_objective = tuning.make_lstm_objective(X_scaled, y_scaled, split_idx, tune_epochs=30)
study_lstm = tuning.run_study(lstm_objective, n_trials=15, seed=42)

print(f"Best params: {study_lstm.best_params}")

tuned_config = models.LSTMConfig(**study_lstm.best_params)
tuned_result = models.train_eval_lstm(top_series_feat, features.BASE_FEATURE_COLS, tuned_config)
print(f"Tuned LSTM WAPE: {tuned_result['wape']:.2f}% (untuned was {baseline_result['wape']:.2f}%)")


## 5. Additional covariates: transactions and a smoother promo signal

Add `transactions_lag1` (that store's daily transaction count, lagged one
day so same-day counts -- only observed after the fact -- don't leak) and
`promo_roll7` (a 7-day rolling mean of promotion activity instead of the
same-day raw count).


In [ ]:
top_series_v1 = features.add_promo_roll7(top_series_feat)
covariate_cols = list(features.BASE_FEATURE_COLS)
if txn_available:
    top_series_v1 = data_mod.add_transactions_lag1(top_series_v1, transactions, top_store)
    covariate_cols = covariate_cols + ["transactions_lag1"]
covariate_cols = covariate_cols + ["promo_roll7"]
top_series_v1 = top_series_v1.dropna(subset=covariate_cols).reset_index(drop=True)

covariate_result = models.train_eval_lstm(top_series_v1, covariate_cols, tuned_config)
print(f"Feature set: {covariate_cols}")
print(f"With extra covariates WAPE: {covariate_result['wape']:.2f}% (tuned baseline was {tuned_result['wape']:.2f}%)")


## 6. Walk-forward backtest

A single 80/20 split can be lucky or unlucky. Retrain across 5
expanding windows instead and look at the spread, not just the mean.


In [ ]:
wf_lstm = backtest.walk_forward_lstm(top_series_v1, covariate_cols, tuned_config, k_folds=5)
print(wf_lstm.to_string(index=False))
print(f"Mean WAPE across folds: {wf_lstm['wape'].mean():.2f}% (std {wf_lstm['wape'].std():.2f})")
print(f"(single-split WAPE was {covariate_result['wape']:.2f}% -- compare to see how much that number")
print(" depends on which slice of time it landed on)")


## 7. XGBoost benchmark

Same feature set, same split, a much cheaper non-sequential model.


In [ ]:
xgb_result = models.train_eval_xgb(top_series_v1, covariate_cols)
print(f"XGBoost WAPE: {xgb_result['wape']:.2f}%")
print(f"LSTM WAPE:    {covariate_result['wape']:.2f}%")

importances = pd.Series(xgb_result["model"].feature_importances_, index=covariate_cols).sort_values(ascending=False)
print()
print("XGBoost feature importances:")
print(importances)


## 8. Does this generalize to other series?

Everything above was tuned on one series (the highest-volume one). Apply
the same architecture and base feature set, with **no per-series
re-tuning**, to 5 more high-volume series and see how much the result
holds up.


In [ ]:
multi_results = []
for (st, fam) in series_list_pooled[1:]:
    ser = data_mod.build_series(train, oil, holidays, st, fam)
    sx = features.engineer_features(ser, features.BASE_FEATURE_COLS)
    if len(sx) < tuned_config.sequence_length + 100:
        continue
    r = models.train_eval_lstm(sx, features.BASE_FEATURE_COLS, tuned_config)
    multi_results.append({"store_nbr": st, "family": fam, "wape": r["wape"]})

multi_df = pd.DataFrame(multi_results)
print(multi_df.to_string(index=False))
print(f"Mean WAPE across {len(multi_df)} additional series: {multi_df['wape'].mean():.2f}% (std {multi_df['wape'].std():.2f})")
print(f"(for reference, the series tuned above scored {tuned_result['wape']:.2f}%)")


## 9. Fixing the holiday feature, and an earthquake shock dummy

The holiday flag used so far only checks *National* holidays. The real
`holidays_events.csv` also carries Regional (state) and Local (city)
holidays, plus a `transferred` flag -- a holiday moved to another date is
NOT a day off on its original date. Rebuild it properly for this store's
own city/state, and add a binary dummy for the ~3-week demand shock that
followed the April 2016 Ecuador earthquake, a well-documented feature of
this dataset.


In [ ]:
store_row = stores[stores["store_nbr"] == top_store].iloc[0]
s2 = data_mod.add_holiday_v2_and_shock(top_series_v1, holidays, store_row["city"], store_row["state"])

print(f"Old (national-only) holiday flag: {int(s2['is_holiday'].sum())} days")
print(f"New (national+regional+local, transfer-aware) flag: {int(s2['is_holiday_v2'].sum())} days")
print(f"Earthquake-window days flagged: {int(s2['earthquake_shock'].sum())}")

feat_base_v2 = [c if c != "is_holiday" else "is_holiday_v2" for c in covariate_cols] + ["earthquake_shock"]
result_holiday_fix = models.train_eval_lstm(s2, feat_base_v2, tuned_config)
print(f"WAPE with corrected holidays + earthquake dummy: {result_holiday_fix['wape']:.2f}%")
print(f"vs. base feature set: {tuned_result['wape']:.2f}%, vs. +covariates: {covariate_result['wape']:.2f}%")


## 10. Covariate ablation

Adding `transactions_lag1` and `promo_roll7` together (Section 5) didn't
clearly help. Add them one at a time on top of the holiday fix to see
whether either one is pulling its own weight, or whether the whole gain
was coming from the holiday fix alone.


In [ ]:
variants = {"holiday_fix_only": list(feat_base_v2)}
if txn_available:
    variants["+transactions_lag1"] = feat_base_v2 + ["transactions_lag1"]
variants["+promo_roll7"] = feat_base_v2 + ["promo_roll7"]
if txn_available:
    variants["+both"] = feat_base_v2 + ["transactions_lag1", "promo_roll7"]

ablation_results = {}
for name, feats in variants.items():
    r = models.train_eval_lstm(s2, feats, tuned_config)
    ablation_results[name] = r
    print(f"{name:24s} WAPE={r['wape']:.2f}%  (n_features={len(feats)})")

best_variant_name = min(ablation_results, key=lambda k: ablation_results[k]["wape"])
print(f"\nBest variant: '{best_variant_name}' at WAPE={ablation_results[best_variant_name]['wape']:.2f}%")


## 11. A second Fourier harmonic

XGBoost's feature importances (Section 7) showed `dow_sin` carrying a
large share of total importance -- a single sin/cos pair can only
represent one harmonic of weekly seasonality. Add a second harmonic for
day-of-week and month on top of whichever ablation variant won.


In [ ]:
s2 = features.add_second_harmonic(s2)
feat_harmonics = variants[best_variant_name] + ["dow_sin2", "dow_cos2", "month_sin2", "month_cos2"]
result_harmonics = models.train_eval_lstm(s2, feat_harmonics, tuned_config)
print(f"WAPE with 2nd harmonic: {result_harmonics['wape']:.2f}% vs '{best_variant_name}' alone: "
      f"{ablation_results[best_variant_name]['wape']:.2f}%")

# Track the best single-series LSTM configuration found across Sections 9-11.
candidates = {**ablation_results, "harmonics": result_harmonics}
best_lstm_name = min(candidates, key=lambda k: candidates[k]["wape"])
best_lstm_result = candidates[best_lstm_name]
best_feat_cols = best_lstm_result["feat_cols"]
best_df = s2
print(f"\nBest LSTM configuration overall: '{best_lstm_name}' at WAPE={best_lstm_result['wape']:.2f}%, "
      f"using {len(best_feat_cols)} features.")


## 12. Tuning XGBoost on the winning feature set

Same idea as Section 4, but for XGBoost on the feature set that won above.


In [ ]:
xgb_objective = tuning.make_xgb_objective(best_df, best_feat_cols, models.train_eval_xgb)
study_xgb = tuning.run_study(xgb_objective, n_trials=30, seed=42)

best_xgb_params = dict(study_xgb.best_params)
best_xgb_params.update(early_stopping_rounds=30, eval_metric="rmse", random_state=42)
best_xgb = models.train_eval_xgb(best_df, best_feat_cols, best_xgb_params)

print(f"Best params: {study_xgb.best_params}")
print(f"Tuned XGBoost WAPE: {best_xgb['wape']:.2f}%")
print(f"Best LSTM ('{best_lstm_name}'): {best_lstm_result['wape']:.2f}%")


## 13. Pooled vs. independent models

Everything so far fits one model per series. Does training a single
model *across* several series -- with store/family metadata as
categorical features so it can borrow strength -- actually beat fitting
each series independently? This uses the plain base feature set so the
pooling question isn't tangled up with the feature-engineering question.


In [ ]:
series_frames = {}
for (st, fam) in series_list_pooled:
    ser = data_mod.build_series(train, oil, holidays, st, fam)
    series_frames[(st, fam)] = features.engineer_features(ser, features.BASE_FEATURE_COLS)

independent_results = []
for key, sx in series_frames.items():
    r = models.train_eval_xgb(sx, features.BASE_FEATURE_COLS)
    independent_results.append({"store_nbr": key[0], "family": key[1], "wape": r["wape"]})
independent_df = pd.DataFrame(independent_results)

pooled_df = pooling.build_pooled_frame(series_frames, stores)
pooled_model, pooled_df, pooled_per_series_df, overall_pooled_wape = pooling.fit_pooled_xgb(
    pooled_df, features.BASE_FEATURE_COLS
)

print("Independent per-series XGBoost:")
print(independent_df.to_string(index=False))
print(f"Mean WAPE (independent): {independent_df['wape'].mean():.2f}%")
print()
print("Pooled (one XGBoost model, all series):")
print(pooled_per_series_df.to_string(index=False))
print(f"Mean WAPE (pooled): {pooled_per_series_df['wape'].mean():.2f}%  |  overall: {overall_pooled_wape:.2f}%")


## 14. Applying the winning feature set to the pooled model

Section 13's pooled model only used the plain base feature set. Rebuild
the Section 9-10 winning feature set (corrected holiday flag -- computed
per series' own city/state, not just the top series' -- earthquake dummy,
`transactions_lag1`, `promo_roll7`) for *every* pooled series, and refit
the pooled model on top of it. This is the step that combines the two
gains instead of leaving them isolated -- and, spoiler, it's also the
step where the single-series "winning" feature set turns out not to
transfer cleanly to the pooled setting.


In [ ]:
winning_feat_template = [c if c != "is_holiday" else "is_holiday_v2" for c in features.BASE_FEATURE_COLS]
winning_feat_template += ["earthquake_shock"]
if txn_available:
    winning_feat_template += ["transactions_lag1"]
winning_feat_template += ["promo_roll7"]

series_frames_v2 = {}
for (st, fam) in series_list_pooled:
    ser = data_mod.build_series(train, oil, holidays, st, fam)
    sx = features.engineer_features(ser, features.BASE_FEATURE_COLS)
    sx = features.add_promo_roll7(sx)
    if txn_available:
        sx = data_mod.add_transactions_lag1(sx, transactions, st)
    row = stores[stores["store_nbr"] == st].iloc[0]
    sx = data_mod.add_holiday_v2_and_shock(sx, holidays, row["city"], row["state"])
    sx = sx.dropna(subset=winning_feat_template).reset_index(drop=True)
    series_frames_v2[(st, fam)] = sx

independent_v2_results = []
for key, sx in series_frames_v2.items():
    r = models.train_eval_xgb(sx, winning_feat_template)
    independent_v2_results.append({"store_nbr": key[0], "family": key[1], "wape": r["wape"]})
independent_v2_df = pd.DataFrame(independent_v2_results)

pooled_df_v2 = pooling.build_pooled_frame(series_frames_v2, stores)
pooled_model_v2, pooled_df_v2, pooled_per_series_df_v2, overall_pooled_wape_v2 = pooling.fit_pooled_xgb(
    pooled_df_v2, winning_feat_template
)

print(f"Independent, winning features: mean WAPE={independent_v2_df['wape'].mean():.2f}%")
print(f"Pooled, winning features:      mean WAPE={pooled_per_series_df_v2['wape'].mean():.2f}% "
      f"(overall {overall_pooled_wape_v2:.2f}%)")
print(f"Pooled, plain feature set (Section 13): {pooled_per_series_df['wape'].mean():.2f}%")


## 15. Walk-forward backtest of the pooled model

Section 14's number reflects a single 80/20 split. Check whether it holds
up with 5 expanding-window folds -- computed independently per series (so
no series' fold boundary leaks into another's calendar), then pooled
fold-by-fold so each fold trains one pooled model across all series and
scores it on the next chunk of every series at once.


In [ ]:
series_frames_for_backtest = {}
for key in series_list_pooled:
    m = (pooled_df_v2["store_nbr"] == key[0]) & (pooled_df_v2["family"] == key[1])
    frame = pooled_df_v2.loc[m].drop(columns=["_split", "pred"], errors="ignore").reset_index(drop=True)
    series_frames_for_backtest[key] = frame

wf_pooled = backtest.walk_forward_pooled_xgb(series_frames_for_backtest, winning_feat_template, n_folds=5)
print(wf_pooled.to_string(index=False))
print(f"Mean WAPE across folds: {wf_pooled['wape'].mean():.2f}% (std {wf_pooled['wape'].std():.2f})")
print(f"vs. single-split: {overall_pooled_wape_v2:.2f}%")


## 16. Tuning the pooled model properly

Sections 13-15 all reused the single-series hand-set XGBoost
hyperparameters for the pooled model too. Tune them directly for the
pooled task with a seeded 30-trial search.


In [ ]:
feat_cols_full = winning_feat_template + pooling.CAT_COLS
train_mask = pooled_df_v2["_split"] == "train"
test_mask = pooled_df_v2["_split"] == "test"
Xp_train = pooled_df_v2.loc[train_mask, feat_cols_full]
yp_train = pooled_df_v2.loc[train_mask, "sales"].values.astype("float32")
Xp_test = pooled_df_v2.loc[test_mask, feat_cols_full]
yp_test = pooled_df_v2.loc[test_mask, "sales"].values.astype("float32")

pooled_xgb_objective = tuning.make_pooled_xgb_objective(Xp_train, yp_train, Xp_test, yp_test)
study_pooled_xgb = tuning.run_study(pooled_xgb_objective, n_trials=30, seed=42)

best_pooled_params = dict(study_pooled_xgb.best_params)
best_pooled_params.update(early_stopping_rounds=30, eval_metric="rmse", random_state=42,
                           enable_categorical=True, tree_method="hist")
_, _, _, overall_pooled_wape_tuned = pooling.fit_pooled_xgb(pooled_df_v2, winning_feat_template, best_pooled_params)

print(f"Best params: {study_pooled_xgb.best_params}")
print(f"Tuned pooled WAPE: {overall_pooled_wape_tuned:.2f}%")
print(f"vs. hand-tuned pooled WAPE: {overall_pooled_wape_v2:.2f}%")


## 17. A fair ensemble comparison

Section 12 left LSTM and XGBoost as separate single-series models. A
naive 50/50 average is the obvious next step, but "obvious" isn't the
same as "best" -- compare it against a weight grid-search and a linear
stacker, all fit on the *first half* of the test period and scored on
the *second half*, so the reported number isn't fit on the same data
it's evaluated on.


In [ ]:
ensemble_result = ensemble.evaluate_ensembles(
    best_lstm_result["preds"], best_xgb["preds"], best_xgb["actual"], fit_frac=0.5
)

for name, score in ensemble_result["scores"].items():
    marker = "  <- best" if name == ensemble_result["best"] else ""
    print(f"{name:16s} 2nd-half WAPE={score:.2f}%{marker}")
print(f"\nGrid-searched weight: {ensemble_result['grid_weight_value']:.2f} (XGB share)")
print(f"Stacker coefficients: {ensemble_result['stacker_coefs']}")


## 18. Results summary

See [`README.md`](./README.md) for the full results table and discussion.
The short version: pooling helps, a properly-corrected holiday feature
helps for a single series but doesn't transfer cleanly once you pool
across series, walk-forward backtesting confirms the pooled result is
reasonably stable over time, Optuna tuning gives a small but real edge,
and -- once scored fairly on held-out data -- the naive 50/50 ensemble
beats both the tuned weight and the linear stacker. The least flattering
results here are the most informative ones.
